# ETL v1dd 04 — Cell-type cluster hex colors

Populate the `hex_color` column of the `Cluster` rows written by
`etl_v1dd_01_cave_dataset_celltypes.ipynb` (hierarchy `v1dd_cell_types`).

The clusters are a 3-level tree: root (`neuron`) → coarse E/I → fine cell type.
This notebook reads those existing clusters, assigns each one a hex color from
the reference palette below, reports any mismatch between the existing cell
types and the supplied palette, and re-writes the `Cluster` rows (the write is
scoped/overwritten per `hierarchy_id`, so parent/children/level are preserved).

In [3]:
from connects_common_connectivity.models import Cluster
from connects_common_connectivity.io import write_models
from connects_common_connectivity.config import get_settings

from pathlib import Path

import pandas as pd
from deltalake import DeltaTable

In [4]:
OUTPUT_ROOT = get_settings().output_root
HIERARCHY_ID = "v1dd_cell_types"
CLUSTER_SUBDIR = "cluster"  # matches connects_common_connectivity WriteSpec for Cluster

cluster_table_path = Path(OUTPUT_ROOT) / CLUSTER_SUBDIR
print(f"output_root       : {OUTPUT_ROOT}")
print(f"cluster table path: {cluster_table_path}")

output_root       : /scratch/v1dd_ccm_20260908
cluster table path: /scratch/v1dd_ccm_20260908/cluster


## Reference palette

RGB values (0–1) provided for each cell type, converted to hex.

In [5]:
cell_type_palette = {
    "L2IT": [0.9075666881074735, 0.7831311441799196, 0.6949858653492867],
    "L3IT": [0.8708834148859907, 0.6888410680873137, 0.6022311567741945],
    "L4IT": [0.8274067443395999, 0.5906550752430283, 0.5233409192850811],
    "L5ET": [0.7795806757764551, 0.5022849626842287, 0.4643377130216367],
    "L5IT": [0.719621740596695, 0.41459615740890743, 0.41383679416754976],
    "L5NP": [0.6539490167887168, 0.3392197573102502, 0.37390086854124416],
    "L6CT": [0.5747615524284997, 0.26764431745066686, 0.3354259029195752],
    "L6IT": [0.4927058890394963, 0.20865645456553394, 0.299493097642672],
    "L6SP": [.4, .2, .2],
    "DTC": [0.5054276262176057, 0.7742578554634272, 0.7550447895194092],
    "ITC": [0.307398668309227, 0.535864603420623, 0.6514437955029915],
    "PTC": [0.21347879954025112, 0.2909729895091856, 0.47951242239001035],
    "STC": [0.12751763609942932, 0.10157769757635066, 0.2292792004813278],
    "glutamatergic": [0.72830947, 0.47437862, 0.46344404],
    "gabaergic": [0.28845568, 0.42566829, 0.52882005],
}


def rgb_to_hex(rgb_list):
    """Convert RGB values (0-1 range) to hex string"""
    # Convert 0-1 range to 0-255 range and then to hex
    r, g, b = [int(val * 255) for val in rgb_list]
    return f"#{r:02x}{g:02x}{b:02x}"


# Convert the entire palette
cell_type_palette_hex = {
    key: rgb_to_hex(rgb_values)
    for key, rgb_values in cell_type_palette.items()
}
cell_type_palette_hex

{'L2IT': '#e7c7b1',
 'L3IT': '#deaf99',
 'L4IT': '#d29685',
 'L5ET': '#c68076',
 'L5IT': '#b76969',
 'L5NP': '#a6565f',
 'L6CT': '#924455',
 'L6IT': '#7d354c',
 'L6SP': '#663333',
 'DTC': '#80c5c0',
 'ITC': '#4e88a6',
 'PTC': '#364a7a',
 'STC': '#20193a',
 'glutamatergic': '#b97876',
 'gabaergic': '#496c86'}

## Read the existing clusters

Read the `Cluster` rows for this hierarchy exactly as written by notebook 01,
so colors are attached to the cell types that actually exist.

In [6]:
cluster_df = DeltaTable(str(cluster_table_path)).to_pandas()
cluster_df = cluster_df[cluster_df["hierarchy_id"] == HIERARCHY_ID].reset_index(drop=True)
print(f"clusters in hierarchy {HIERARCHY_ID!r}: {len(cluster_df)}")
cluster_df[["id", "parent", "level"]].sort_values(["level", "id"])

clusters in hierarchy 'v1dd_cell_types': 15


,id,parent,level
0,neuron,None,0
1,E,neuron,1
2,I,neuron,1
3,DTC,I,2
4,ITC,I,2
5,L2-IT,E,2
6,L3-IT,E,2
7,L4-IT,E,2
8,L5-ET,E,2
9,L5-IT,E,2


## Map cluster ids → hex color

The palette keys and the cluster ids use slightly different spellings, so we
normalize before matching:

- Fine excitatory ids carry a dash (`L2-IT`) while the palette omits it
  (`L2IT`); normalization strips `-`/`_` and lowercases both sides.
- Fine inhibitory ids (`DTC`, `ITC`, `PTC`, `STC`) match the palette directly.
- Coarse ids `E` / `I` map to the palette's `glutamatergic` / `gabaergic`.
- The root id `neuron` has no entry in the palette and is left uncolored.

Anything else is reported as a mismatch rather than silently skipped.

In [7]:
COARSE_ALIASES = {"E": "glutamatergic", "I": "gabaergic"}


def normalize_key(s: str) -> str:
    return s.replace("-", "").replace("_", "").lower()


palette_norm = {normalize_key(k): k for k in cell_type_palette_hex}

used_palette_keys = set()


def resolve_hex(cluster_id: str):
    """Return (hex_color, palette_key) for a cluster id, or (None, None)."""
    if cluster_id in COARSE_ALIASES:
        key = COARSE_ALIASES[cluster_id]
        used_palette_keys.add(key)
        return cell_type_palette_hex[key], key
    key = palette_norm.get(normalize_key(cluster_id))
    if key is not None:
        used_palette_keys.add(key)
        return cell_type_palette_hex[key], key
    return None, None


cluster_df["hex_color"], cluster_df["palette_key"] = zip(
    *cluster_df["id"].map(resolve_hex)
)
cluster_df[["id", "level", "palette_key", "hex_color"]].sort_values(["level", "id"])

,id,level,palette_key,hex_color
0,neuron,0,None,None
1,E,1,glutamatergic,#b97876
2,I,1,gabaergic,#496c86
3,DTC,2,DTC,#80c5c0
4,ITC,2,ITC,#4e88a6
5,L2-IT,2,L2IT,#e7c7b1
6,L3-IT,2,L3IT,#deaf99
7,L4-IT,2,L4IT,#d29685
8,L5-ET,2,L5ET,#c68076
9,L5-IT,2,L5IT,#b76969


### Mismatch report

- **Uncolored clusters**: cluster ids with no palette match. `neuron` (the root)
  is expected here; anything else is a genuine mismatch to investigate.
- **Unused palette keys**: palette entries that did not match any cluster.

In [8]:
EXPECTED_UNCOLORED = {"neuron"}

uncolored = sorted(cluster_df.loc[cluster_df["hex_color"].isna(), "id"])
unused_palette = sorted(set(cell_type_palette_hex) - used_palette_keys)

unexpected_uncolored = sorted(set(uncolored) - EXPECTED_UNCOLORED)

print("Uncolored clusters      :", uncolored)
print("  expected uncolored    :", sorted(EXPECTED_UNCOLORED & set(uncolored)))
print("  UNEXPECTED uncolored  :", unexpected_uncolored)
print("Unused palette keys     :", unused_palette)

if unexpected_uncolored:
    print("\n\u26a0\ufe0f  cluster ids with no matching palette color:", unexpected_uncolored)
if unused_palette:
    print("\u26a0\ufe0f  palette keys that matched no cluster       :", unused_palette)
if not unexpected_uncolored and not unused_palette:
    print("\n\u2705  All cell types matched a palette color and all palette keys were used.")

Uncolored clusters      : ['neuron']
  expected uncolored    : ['neuron']
  UNEXPECTED uncolored  : []
Unused palette keys     : ['L6SP']
⚠️  palette keys that matched no cluster       : ['L6SP']


## Write the colored clusters back

Reconstruct `Cluster` models from the existing rows (preserving `parent`,
`children`, and `level`) with `hex_color` populated, then write. The Cluster
write is overwrite-scoped by `hierarchy_id`, so this replaces the hierarchy's
rows in place.

In [9]:
def to_children(value):
    """Normalize the stored children column to a list[str] or None."""
    if value is None:
        return None
    # deltalake returns list columns as numpy arrays / lists
    items = list(value)
    return [str(x) for x in items] if len(items) else None


def to_opt_int(value):
    return None if pd.isna(value) else int(value)


def to_opt_str(value):
    return None if value is None or (isinstance(value, float) and pd.isna(value)) else str(value)


colored_clusters = [
    Cluster(
        id=str(row.id),
        hierarchy_id=HIERARCHY_ID,
        parent=to_opt_str(row.parent),
        children=to_children(row.children),
        level=to_opt_int(row.level),
        hex_color=(None if pd.isna(row.hex_color) else str(row.hex_color)),
    )
    for row in cluster_df.itertuples()
]

result = write_models(colored_clusters, output_root=OUTPUT_ROOT)
n_colored = sum(c.hex_color is not None for c in colored_clusters)
print(f"Cluster rows written: {result.rows_written} "
      f"({n_colored}/{len(colored_clusters)} with a hex color)")

Cluster rows written: 15 (14/15 with a hex color)


## Verify

Re-read the cluster table and confirm the colors landed.

In [10]:
verify_df = DeltaTable(str(cluster_table_path)).to_pandas()
verify_df = verify_df[verify_df["hierarchy_id"] == HIERARCHY_ID]
verify_df[["id", "level", "parent", "hex_color"]].sort_values(["level", "id"]).reset_index(drop=True)

,id,level,parent,hex_color
0,neuron,0,None,None
1,E,1,neuron,#b97876
2,I,1,neuron,#496c86
3,DTC,2,I,#80c5c0
4,ITC,2,I,#4e88a6
5,L2-IT,2,E,#e7c7b1
6,L3-IT,2,E,#deaf99
7,L4-IT,2,E,#d29685
8,L5-ET,2,E,#c68076
9,L5-IT,2,E,#b76969
